# **Data Loading**

In [ ]:
!pip install torch transformers scikit-learn --quiet

import os, re, json, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
CFG = {
    "model_name":   "bert-base-uncased",
    "max_length":   256,
    "batch_size":   16,
    "epochs":       10,       # was 3
    "lr":           1e-5,    # was 2e-5
    "warmup_ratio": 0.2,     # was 0.1
    "weight_decay": 0.01,
    "patience":     2,
    "seed":         42,
    "val_size":     0.1,
    "test_size":    0.1,
    "save_dir":     "./bert_newsgroups_model_v2",  # new dir, clean run
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
}

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"Device: {CFG['device']}")

Device: cuda


In [ ]:
raw = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),  # strip metadata, keep body
    shuffle=True,
    random_state=CFG["seed"],
)
texts      = raw.data
labels     = np.array(raw.target)
class_names = raw.target_names
num_labels  = len(class_names)
print(f"{len(texts)} documents | {num_labels} classes")

# 80 / 10 / 10 split
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=CFG["seed"]
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=CFG["seed"]
)
print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

18846 documents | 20 classes
Train: 15076 | Val: 1885 | Test: 1885


In [ ]:
def clean_text(text: str) -> str:
    #  DO: normalise whitespace, strip non-ASCII noise
    #  DON'T: lowercase, remove stopwords, lemmatise, strip punctuation
    text = re.sub(r"[^\x00-\x7F]+", " ", text)  # remove non-ASCII
    text = re.sub(r"\s+", " ", text).strip()      # collapse whitespace
    return text

train_texts = [clean_text(t) for t in train_texts]
val_texts   = [clean_text(t) for t in val_texts]
test_texts  = [clean_text(t) for t in test_texts]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=CFG["max_length"],
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "token_type_ids": self.encodings["token_type_ids"][idx],
            "labels":         self.labels[idx],
        }

train_ds = NewsDataset(train_texts, train_labels)
val_ds   = NewsDataset(val_texts,   val_labels)
test_ds  = NewsDataset(test_texts,  test_labels)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],   shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"]*2, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"]*2, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=num_labels,
    hidden_dropout_prob=0.3,           # was 0.1
    attention_probs_dropout_prob=0.2,  # was 0.1
    classifier_dropout=0.3,            # was not set
)
model = model.to(CFG["device"])
print(f"Loaded: {CFG['model_name']} → {num_labels} classes")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded: bert-base-uncased → 20 classes


In [ ]:
total_steps  = len(train_loader) * CFG["epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

# Exclude biases and LayerNorm from weight decay
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_params = [
    {
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": CFG["weight_decay"],
    },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(optimizer_grouped_params, lr=CFG["lr"], eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

Total steps: 4715 | Warmup steps: 943


In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(CFG["device"])
            mask  = batch["attention_mask"].to(CFG["device"])
            ttype = batch["token_type_ids"].to(CFG["device"])
            lbls  = batch["labels"].to(CFG["device"])

            out  = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype, labels=lbls)
            loss = out.loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent exploding grads
                optimizer.step()
                scheduler.step()

            correct    += (out.logits.argmax(-1) == lbls).sum().item()
            total      += lbls.size(0)
            total_loss += loss.item() * lbls.size(0)

    return total_loss / total, correct / total


best_val_loss, best_weights, patience_count = float("inf"), None, 0
history = []

for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val_loss={vl_loss:.4f} acc={vl_acc:.4f} | {time.time()-t0:.0f}s")

    history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_acc=tr_acc,
                        vl_loss=vl_loss, vl_acc=vl_acc))

    if vl_loss < best_val_loss:
        best_val_loss  = vl_loss
        best_weights   = copy.deepcopy(model.state_dict())
        patience_count = 0
        print("  ✓ Best model updated")
    else:
        patience_count += 1
        if patience_count >= CFG["patience"]:
            print("  Early stopping triggered.")
            break

model.load_state_dict(best_weights)  # restore best checkpoint

Epoch 1 | train_loss=2.6109 acc=0.2349 | val_loss=1.6391 acc=0.5496 | 739s
  ✓ Best model updated
Epoch 2 | train_loss=1.3948 acc=0.6148 | val_loss=1.1164 acc=0.6711 | 757s
  ✓ Best model updated
Epoch 3 | train_loss=1.0751 acc=0.6852 | val_loss=0.9889 acc=0.7013 | 755s
  ✓ Best model updated
Epoch 4 | train_loss=0.9462 acc=0.7169 | val_loss=0.9511 acc=0.7162 | 756s
  ✓ Best model updated
Epoch 5 | train_loss=0.8802 acc=0.7392 | val_loss=0.9465 acc=0.7183 | 756s
  ✓ Best model updated


<All keys matched successfully>

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids   = batch["input_ids"].to(CFG["device"])
        mask  = batch["attention_mask"].to(CFG["device"])
        ttype = batch["token_type_ids"].to(CFG["device"])
        lbls  = batch["labels"].to(CFG["device"])

        logits = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype).logits
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_true.extend(lbls.cpu().numpy())

print(f"Test Accuracy: {accuracy_score(all_true, all_preds):.4f}\n")
print(classification_report(all_true, all_preds, target_names=class_names, digits=3))

Test Accuracy: 0.7183

                          precision    recall  f1-score   support

             alt.atheism      0.481     0.463     0.471        80
           comp.graphics      0.680     0.701     0.690        97
 comp.os.ms-windows.misc      0.688     0.653     0.670        98
comp.sys.ibm.pc.hardware      0.556     0.714     0.625        98
   comp.sys.mac.hardware      0.764     0.567     0.651        97
          comp.windows.x      0.856     0.838     0.847        99
            misc.forsale      0.852     0.704     0.771        98
               rec.autos      0.539     0.828     0.653        99
         rec.motorcycles      0.843     0.590     0.694       100
      rec.sport.baseball      0.872     0.828     0.850        99
        rec.sport.hockey      0.912     0.930     0.921       100
               sci.crypt      0.698     0.818     0.753        99
         sci.electronics      0.696     0.788     0.739        99
                 sci.med      0.854     0.889     0.

In [ ]:
os.makedirs(CFG["save_dir"], exist_ok=True)

model.save_pretrained(CFG["save_dir"])
tokenizer.save_pretrained(CFG["save_dir"])

# label map — critical for deployment, keeps labels in sync with model
label_map = {i: name for i, name in enumerate(class_names)}
with open(f"{CFG['save_dir']}/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

with open(f"{CFG['save_dir']}/train_config.json", "w") as f:
    json.dump({**CFG, "device": str(CFG["device"])}, f, indent=2)

print(f"Saved to {CFG['save_dir']}/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./bert_newsgroups_model_v2/


In [ ]:
checkpoint = {
    "model_state_dict":     model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "best_val_loss":        best_val_loss,
    "history":              history,
    "config":               CFG,
    "num_labels":           num_labels,
    "class_names":          class_names,
}

pth_path = os.path.join(CFG["save_dir"], "bert_newsgroups.pth")
torch.save(checkpoint, pth_path)
print(f"Checkpoint saved → {pth_path}")

Checkpoint saved → ./bert_newsgroups_model_v2/bert_newsgroups.pth


# **Retrainig**

In [ ]:
# ── Resume Training ──────────────────────────────────────

# 1. Rebuild model
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=num_labels,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.2,
    classifier_dropout=0.3,
)
model = model.to(CFG["device"])

# 2. Rebuild optimizer & scheduler
total_steps  = len(train_loader) * CFG["epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_params = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],     "weight_decay": 0.0},
]
optimizer = AdamW(optimizer_grouped_params, lr=CFG["lr"], eps=1e-8)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# 3. Load checkpoint
checkpoint = torch.load("/content/bert_newsgroups_model_v2/bert_newsgroups.pth", map_location=CFG["device"])
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
history        = checkpoint["history"]
best_val_loss  = checkpoint["best_val_loss"]
best_weights   = copy.deepcopy(model.state_dict())
patience_count = 0
epochs_done    = len(history)
print(f"Resumed from epoch {epochs_done} | best_val_loss={best_val_loss:.4f}")

# 4. Continue training
EXTRA_EPOCHS = 5  # how many more epochs to run

for epoch in range(epochs_done + 1, epochs_done + EXTRA_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    print(f"Epoch {epoch} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val_loss={vl_loss:.4f} acc={vl_acc:.4f} | {time.time()-t0:.0f}s")

    history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_acc=tr_acc,
                        vl_loss=vl_loss, vl_acc=vl_acc))

    if vl_loss < best_val_loss:
        best_val_loss  = vl_loss
        best_weights   = copy.deepcopy(model.state_dict())
        patience_count = 0
        print("  ✓ Best model updated")
    else:
        patience_count += 1
        print(f"  No improvement ({patience_count}/{CFG['patience']})")
        if patience_count >= CFG["patience"]:
            print("  Early stopping triggered.")
            break

model.load_state_dict(best_weights)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Resumed from epoch 5 | best_val_loss=0.9465
Epoch 6 | train_loss=0.8594 acc=0.7452 | val_loss=0.9465 acc=0.7183 | 749s
  No improvement (1/2)
Epoch 7 | train_loss=0.8602 acc=0.7448 | val_loss=0.9465 acc=0.7183 | 758s
  No improvement (2/2)
  Early stopping triggered.


<All keys matched successfully>